**Create some fake data**

In [1]:
from system.data_classes import *
from faker import Faker


Faker.seed(42)
fake = Faker()

number_of_tuples = 10000

# create some fake data mimicking tuples in a relation
books = [
    Book(
        fake.sentence(),
        fake.name(),
        fake.pyfloat(min_value=0, max_value=100, right_digits=2),
    )
    for i in range(number_of_tuples)
]
books[:10]

[Book(title='Agent every development say.', author='Stephanie Miller', price=54.4),
 Book(title='Behavior discussion own.', author='Michelle Miles', price=53.28),
 Book(title='Operation speak according south recently future choice.', author='Gabrielle Davis', price=12.45),
 Book(title='Page southern role movie win her.', author='Sandra Montgomery', price=46.73),
 Book(title='Relate animal direction eye bag.', author='Brian Ramirez', price=81.46),
 Book(title='Have decide environment view.', author='Lisa Jackson', price=93.31),
 Book(title='Left establish understand read.', author='Frederick Tate', price=7.29),
 Book(title='Source husband at.', author='Latoya Robbins', price=27.83),
 Book(title='How trip learn enter east no enjoy.', author='Stephanie Ross', price=28.17),
 Book(title='Management sense technology check civil quite others.', author='Kyle Mcdonald', price=32.7)]

In [2]:
import pandas as pd


# store into pandas dataframe
columns = ["title", "author", "price"]
df = pd.DataFrame(data=books, columns=columns)

**Emulate row store**

To emulate a row-major layout, we write the data into a CSV file where each tuple is stored consecutively. Within each tuple, each attribute is also stored consecutively. Inspect the generated `row.csv` file for more detail.

In [3]:
# write data into file in row-major layout to emulate store
path_row = "row.csv"
df.to_csv(path_or_buf=path_row, sep="\n", header=False, index=False, mode="w")

**Emulate column store**

To emulate a column-major layout, we write the data into a CSV file where each attribute column is stored consecutively. Inspect the generated `column.csv` file for more detail.

In [4]:
# write data into file in column-major layout to emulate store
path_column = "column.csv"
for i, col in enumerate(df.columns):
    df.to_csv(
        path_or_buf=path_column,
        columns=[col],
        header=False,
        index=False,
        mode=(
            "w" if i == 0 else "a"
        ),  # truncate file before first column, append afterward
    )

**Consider the following query accessing multiple attributes**
``` sql
SELECT author, price 
FROM books
LIMIT 314
OFFSET 100
```

This query prints a subset of tuples (only 314 tuple skipping the first 100 tuples) and a subset of attributes (only author and price). However, for each print call, we need multiple attribute values of the *same* tuple. In theory, we expect the row-major layout to be superior as these attribute values are stored consecutively in the row store. So let's measure the actual query execution times! 

Note that our query engine basically loads the needed values again in a pandas dataframe in the given layout. Afterward, the values forming a single tuple has to be accessed (consecutive accesses for row-major layout, random accesses for column-major layout). Each constructed tuple is then printed.

In [5]:
from time import time_ns


attr_idxs = [1, 2]
limit = 314
offset = 100

row_start = time_ns()

# execute query given row-major layout
data = pd.read_csv(
    filepath_or_buffer=path_row,
    header=None,
    skiprows=lambda x: x < offset * len(columns)  # before offset
    or x % len(columns) not in attr_idxs,  # wrong attribute
    nrows=limit * len(attr_idxs),  # do not exceed limit
)
for i in range(0, len(attr_idxs) * limit, len(attr_idxs)):
    print(
        *[data.iloc[i + j][0] for j in range(len(attr_idxs))], sep=", "
    )  # construct tuple by consecutive accesses

row_end = time_ns()

print("Time row-major:", (row_end - row_start) / 1000, "ms")

Victor Brown, 61.51
Donald Schultz, 70.91
Matthew Rich, 70.92
Emily Edwards, 32.47
Patrick Watson, 16.71
Maurice Christensen, 92.6
Brian Bruce, 47.14
Maria Johnson, 84.88
Charles Sandoval, 8.18
Christopher Collins, 41.99
Lawrence Perry, 27.96
Kyle Smith, 38.27
Carl Patel, 29.64
Laura Berry, 90.81
Heidi Kim, 13.31
Jordan Wilson, 65.24
Christian Malone, 9.22
Lori Hernandez, 8.6
Jacob Mitchell, 82.11
Larry Mason, 7.57
Amanda Zimmerman, 56.67
Felicia Rivera, 71.86
Brenda Bailey, 83.89
Michael Valentine, 87.67
Daniel Meza, 62.24
Nancy Rose, 84.43
Michael Young, 56.78
Stephanie Lee, 86.73
Brenda Wright, 82.61
Taylor Hernandez, 55.11
James Miller, 28.3
Melissa Taylor, 90.6
Ricardo Young, 15.6
Alexis Robertson, 30.91
Michael Mills, 46.8
Paul Kelly, 8.44
Joshua Li, 22.99
Rick Blackwell, 40.39
Matthew Nelson, 47.55
Kelly Owen, 93.54
Garrett Phillips, 57.77
Mark Moody, 7.9
Amanda Jones, 99.82
Billy Taylor, 46.5
Deborah Green, 57.27
Michael Bruce, 40.85
Carrie Ward, 28.15
William Anderson, 11.8
Be

In [6]:
column_start = time_ns()

# execute query given column-major layout
data = pd.read_csv(
    filepath_or_buffer=path_column,
    header=None,
    skiprows=lambda x: x % number_of_tuples < offset  # before offset
    or x % number_of_tuples >= offset + limit  # after limit in column
    or int(x / number_of_tuples) not in attr_idxs,  # wrong attribute
    nrows=limit * len(attr_idxs),  # do not exceed limit
)
for i in range(limit):
    print(
        *[data.iloc[j * limit + i][0] for j in range(len(attr_idxs))], sep=", "
    )  # construct tuple by random accesses

column_end = time_ns()

print("Time column-major:", (column_end - column_start) / 1000, "ms")

Victor Brown, 61.51
Donald Schultz, 70.91
Matthew Rich, 70.92
Emily Edwards, 32.47
Patrick Watson, 16.71
Maurice Christensen, 92.6
Brian Bruce, 47.14
Maria Johnson, 84.88
Charles Sandoval, 8.18
Christopher Collins, 41.99
Lawrence Perry, 27.96
Kyle Smith, 38.27
Carl Patel, 29.64
Laura Berry, 90.81
Heidi Kim, 13.31
Jordan Wilson, 65.24
Christian Malone, 9.22
Lori Hernandez, 8.6
Jacob Mitchell, 82.11
Larry Mason, 7.57
Amanda Zimmerman, 56.67
Felicia Rivera, 71.86
Brenda Bailey, 83.89
Michael Valentine, 87.67
Daniel Meza, 62.24
Nancy Rose, 84.43
Michael Young, 56.78
Stephanie Lee, 86.73
Brenda Wright, 82.61
Taylor Hernandez, 55.11
James Miller, 28.3
Melissa Taylor, 90.6
Ricardo Young, 15.6
Alexis Robertson, 30.91
Michael Mills, 46.8
Paul Kelly, 8.44
Joshua Li, 22.99
Rick Blackwell, 40.39
Matthew Nelson, 47.55
Kelly Owen, 93.54
Garrett Phillips, 57.77
Mark Moody, 7.9
Amanda Jones, 99.82
Billy Taylor, 46.5
Deborah Green, 57.27
Michael Bruce, 40.85
Carrie Ward, 28.15
William Anderson, 11.8
Be

We can observe that the row-major layout indeed favors the given query accessing multiple attributes.

**Consider the following query accessing and aggregating a single attribute**
``` sql
SELECT MAX(price) 
FROM books
```

This query aggregates over a single attribute (price). Therefore, we now expect the column-major layout to be superior as all attribute values are stored consecutively in the column store. So again let's measure the actual query execution times! 

Note that our query engine still loads the needed values in a pandas dataframe (random accesses for row-major layout, consecutive accesses for column-major layout). Afterward, all loaded values are aggregated and the result is then printed.

In [7]:
attr_idx = 2

row_start = time_ns()

# execute query given row-major layout
data = pd.read_csv(
    filepath_or_buffer=path_row,
    header=None,
    skiprows=lambda x: x % len(columns) != attr_idx,  # wrong attribute
    nrows=number_of_tuples,
)
print(data.max(0)[0])  # aggregate

row_end = time_ns()

print("Time row-major:", (row_end - row_start) / 1000, "ms")

99.98
Time row-major: 4814.0 ms


In [8]:
column_start = time_ns()

# execute query given column-major layout
data = pd.read_csv(
    filepath_or_buffer=path_column,
    header=None,
    skiprows=attr_idx * number_of_tuples,  # start at attribute column
    nrows=number_of_tuples,
)
print(data.max(0)[0])  # aggregate

column_end = time_ns()

print("Time column-major:", (column_end - column_start) / 1000, "ms")

99.98
Time column-major: 2683.0 ms


In [9]:
!rm row.csv column.csv

We can observe that now the column-major layout indeed favors the given query accessing and aggregating a single attribute.  Note that we could additionally speed up the execution using the column-major layout by utilizing vectorization and SIMD instructions.